In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [8]:
!rm -rf /kaggle/working/processed_age_dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
import numpy as np
from PIL import Image
import requests

# --- 1. SOTA PREPROCESSING (Matched for Live Interface Parity) ---
def enhance_age_prep(input_path, output_path):
    """CLAHE + Brightness + Letterbox Padding (NO BLUR!)"""
    try:
        img = cv2.imread(input_path)
        if img is None: return

        # 1. CLAHE & Brightness (Makes wrinkles/texture pop)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        merged = cv2.merge((clahe.apply(l), a, b))
        contrast_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
        bright_img = cv2.convertScaleAbs(contrast_img, alpha=1.1, beta=15)

        # 2. Letterbox Padding (Directly on the sharp image)
        h, w = bright_img.shape[:2]
        longest_edge = max(h, w)
        top = (longest_edge - h) // 2
        bottom = longest_edge - h - top
        left = (longest_edge - w) // 2
        right = longest_edge - w - left
        
        padded_img = cv2.copyMakeBorder(
            bright_img, top, bottom, left, right, 
            cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )

        cv2.imwrite(output_path, padded_img)
    except Exception as e:
        pass

# IMPORTANT: Update this path to your newly uploaded AGE dataset!
raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/dataset_age_v1/dataset_age_v1' 
processed_base_dir = '/kaggle/working/processed_age_dataset'

print("✨ Phase 1: Applying Preprocessing to Age Dataset...")
# The 5 Age Buckets
AGE_GROUPS = ["kids", "teens", "young_adults", "middle_aged", "seniors"]

for split in ['train', 'val']:
    for group in AGE_GROUPS:
        in_folder = os.path.join(raw_base_dir, split, group)
        out_folder = os.path.join(processed_base_dir, split, group)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{group}"):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                if not os.path.exists(out_path):
                    enhance_age_prep(in_path, out_path)

print("✅ Preprocessing complete! Handing over to PyTorch...")

# --- 2. CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    img = cv2.imread(path)
    if img is None: return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. HYPERPARAMETERS ---
BATCH_SIZE = 64
EPOCHS = 15  # Age is harder than Gender, giving it 15 epochs
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Age Detection V1.0...")

# --- 4. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # NO MORE CROPPING! Keep the hairlines!
    transforms.RandomRotation(15), # Gentle head tilts instead
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Classes detected by PyTorch: {train_dataset.class_to_idx}")

# --- 5. THE ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features
# THE FIX: We now have 5 output classes, not 2!
model.head = nn.Linear(in_features, 5) 
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss() 
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05) 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    
    scheduler.step()
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 AGE VERSION 1.0 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save the Age Brain
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_v1_2_age.pth')
print("💾 Saved weights as 'swin_v1_2_age.pth'")

✨ Phase 1: Applying Preprocessing to Age Dataset...


Processing train/kids:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/teens:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/young_adults:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/middle_aged:   0%|          | 0/944 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/19 [00:00<?, ?it/s]

Train Loss: 0.4663 | Train Acc: 0.8093
Val Loss:   0.6613 | Val Acc:   0.7263

Epoch 15/15
---------------


Training:   0%|          | 0/74 [00:00<?, ?it/s]

Testing:   0%|          | 0/19 [00:00<?, ?it/s]

Train Loss: 0.4622 | Train Acc: 0.8172
Val Loss:   0.6585 | Val Acc:   0.7280

🎉 AGE VERSION 1.0 TRAINING COMPLETE in 20m 45s
🏆 Best Validation Accuracy: 0.7364
💾 Saved weights as 'swin_v1_2_age.pth'


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import copy
import time
import os
import cv2
import numpy as np
from PIL import Image
import requests

# --- 1. SOTA PREPROCESSING ---
def enhance_age_prep(input_path, output_path):
    try:
        img = cv2.imread(input_path)
        if img is None: return

        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        merged = cv2.merge((clahe.apply(l), a, b))
        contrast_img = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
        bright_img = cv2.convertScaleAbs(contrast_img, alpha=1.1, beta=15)

        h, w = bright_img.shape[:2]
        longest_edge = max(h, w)
        top = (longest_edge - h) // 2
        bottom = longest_edge - h - top
        left = (longest_edge - w) // 2
        right = longest_edge - w - left
        
        padded_img = cv2.copyMakeBorder(bright_img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0, 0, 0])
        cv2.imwrite(output_path, padded_img)
    except Exception as e:
        pass

raw_base_dir = '/kaggle/input/datasets/aryannaik225/clean-dataset-1000/dataset_age_v1/dataset_age_v1' 
processed_base_dir = '/kaggle/working/processed_age_dataset'

print("✨ Phase 1: Applying Preprocessing & Enforcing Ordinal Sorting...")

# THE FIX: Force PyTorch to map ages in the correct chronological order
AGE_MAPPING = {
    "kids": "00_kids",
    "teens": "01_teens",
    "young_adults": "02_young_adults",
    "middle_aged": "03_middle_aged",
    "seniors": "04_seniors"
}

for split in ['train', 'val']:
    for raw_group, sorted_group in AGE_MAPPING.items():
        in_folder = os.path.join(raw_base_dir, split, raw_group)
        out_folder = os.path.join(processed_base_dir, split, sorted_group)
        os.makedirs(out_folder, exist_ok=True)
        
        if os.path.exists(in_folder):
            images = os.listdir(in_folder)
            for filename in tqdm(images, desc=f"Processing {split}/{sorted_group}", leave=False):
                in_path = os.path.join(in_folder, filename)
                out_path = os.path.join(out_folder, filename)
                if not os.path.exists(out_path):
                    enhance_age_prep(in_path, out_path)

print("✅ Preprocessing complete! Handing over to PyTorch...")

# --- 2. CUSTOM OPENCV LOADER ---
def custom_opencv_loader(path):
    img = cv2.imread(path)
    if img is None: return Image.new('RGB', (224, 224), (0, 0, 0))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img)

# --- 3. HYPERPARAMETERS ---
BATCH_SIZE = 64
EPOCHS = 30  
LEARNING_RATE = 3e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Firing up PyTorch on {DEVICE.upper()} for Ordinal Age Detection V2.0...")

# --- 4. DATA AUGMENTATION ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15), 
    transforms.RandomHorizontalFlip(),                   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'train'), transform=train_transforms, loader=custom_opencv_loader)
val_dataset = datasets.ImageFolder(os.path.join(processed_base_dir, 'val'), transform=val_transforms, loader=custom_opencv_loader)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"🗂️ Chronological Classes detected: {train_dataset.class_to_idx}")

# --- 5. ORDINAL ENGINE & LOOP ---
model = models.swin_t(weights='DEFAULT')
in_features = model.head.in_features

# ORDINAL FIX: 5 Age Buckets means we only need 4 Binary Thresholds!
model.head = nn.Linear(in_features, 4) 
model = model.to(DEVICE)

# ORDINAL FIX: Binary Cross Entropy replaces standard Cross Entropy
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05) 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
start_time = time.time()

# This tensor helps convert a single integer label (e.g., 2) into [1, 1, 0, 0]
levels = torch.arange(4).to(DEVICE)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    
    model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # --- ORDINAL MATH MAGIC ---
        # Converts standard label '2' into a [1.0, 1.0, 0.0, 0.0] tensor
        ordinal_labels = (labels.unsqueeze(1) > levels).float()
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        # Calculate loss against the 4 binary answers
        loss = criterion(outputs, ordinal_labels)
        
        # Convert the 4 raw binary answers back into a single bucket integer (0-4)
        preds = (torch.sigmoid(outputs) > 0.5).sum(dim=1)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    
    scheduler.step()
        
    train_loss = running_loss / len(train_dataset)
    train_acc = running_corrects.double() / len(train_dataset)
    
    model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            ordinal_labels = (labels.unsqueeze(1) > levels).float()
            
            outputs = model(inputs)
            loss = criterion(outputs, ordinal_labels)
            
            # Predict by summing up the "Yes" answers
            preds = (torch.sigmoid(outputs) > 0.5).sum(dim=1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_loss = val_loss / len(val_dataset)
    val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

time_elapsed = time.time() - start_time
print("\n" + "="*50)
print(f"🎉 ORDINAL AGE V2.0 TRAINING COMPLETE in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_acc:.4f}")
print("="*50)

# Save the Ordinal Age Brain
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), '/kaggle/working/swin_ordinal_age_v3.pth')
print("💾 Saved weights as 'swin_ordinal_age_v3.pth'")

✨ Phase 1: Applying Preprocessing & Enforcing Ordinal Sorting...


Processing train/00_kids:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/01_teens:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/02_young_adults:   0%|          | 0/944 [00:00<?, ?it/s]

Processing train/03_middle_aged:   0%|          | 0/944 [00:00<?, ?it/s]

...

Testing:   0%|          | 0/19 [00:00<?, ?it/s]

Train Loss: 0.0814 | Train Acc: 0.8771
Val Loss:   0.1920 | Val Acc:   0.7381

Epoch 30/30
---------------


Training:   0%|          | 0/74 [00:00<?, ?it/s]

Testing:   0%|          | 0/19 [00:00<?, ?it/s]

Train Loss: 0.0784 | Train Acc: 0.8822
Val Loss:   0.1919 | Val Acc:   0.7390

🎉 ORDINAL AGE V2.0 TRAINING COMPLETE in 37m 10s
🏆 Best Validation Accuracy: 0.7398
💾 Saved weights as 'swin_ordinal_age_v3.pth'
